# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signals checked first, before coding anything:**

1. **Staleness -> decline** (behind FlyRank's refresh flags): does a page's days-since-update relate to whether it's currently declining?
2. **Position -> CTR** (behind the CTR-fix logic): does click-through rate actually fall as position gets worse, the way the CTR-fix reasoning assumes?

Bucket tables with `n` for both are in the code cell below, each with a one-word verdict.

**The rule, in plain words:** a page is worth reviewing for refresh if it still has real search demand (`impressions_90d >= 500`) *and* it hasn't been touched in a long time (`days_since_last_update >= 180`). Score it by how much exposure it has, so among equally-stale-and-visible pages, the ones with more traffic at stake rank first.

**Reason code:** `stale_but_visible` — the only reason code this rule ever emits, on purpose (one rule, one reason, per the assignment). Rows that don't meet both conditions get no reason code and `action = "no_action_needed"`.

**Action:** `review_for_refresh`.

**No leakage, by construction:** the score only reads `days_since_last_update` and `impressions_90d` — both knowable before any "is this page declining" question is asked. `trend_direction` / `trend_pct` never enter the score; they're used below only to check the staleness signal, exactly like Week 1-3's rule.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- Signal 1: staleness -> decline (behind the refresh flags) ---
staleness_bins = [-1, 89, 179, 364, 10_000]
staleness_labels = ["<90d", "90-179d", "180-364d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=staleness_bins, labels=staleness_labels)

staleness_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean"),
).round(3)
print("Signal 1 -- staleness bucket vs decline rate:")
print(staleness_table)

rates = staleness_table["decline_rate"].values
if all(rates[i] <= rates[i + 1] + 0.01 for i in range(len(rates) - 1)) and rates[-1] > rates[0]:
    verdict_1 = "CONFIRMED"
elif all(rates[i] >= rates[i + 1] - 0.01 for i in range(len(rates) - 1)) and rates[-1] < rates[0]:
    verdict_1 = "OPPOSITE"
elif max(rates) - min(rates) < 0.03:
    verdict_1 = "FALSE"
else:
    verdict_1 = "MIXED"
print(f"Verdict: {verdict_1}\n")

# --- Signal 2: position -> CTR (behind the CTR-fix logic) ---
visible = df[df["impressions_90d"] >= 100].copy()
tier_order = ["page_1", "top_3", "striking", "page_3_5", "deep"]
ctr_table = visible.groupby("position_tier", observed=True).agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean"),
).reindex(tier_order).round(3)
print("Signal 2 -- position tier vs mean CTR (impressions_90d >= 100 only):")
print(ctr_table)

ctr_vals = ctr_table["mean_ctr"].dropna().values
verdict_2 = "CONFIRMED" if all(ctr_vals[i] >= ctr_vals[i + 1] for i in range(len(ctr_vals) - 1)) else "MIXED"
print(f"Verdict: {verdict_2}")

Signal 1 -- staleness bucket vs decline rate:
                      n  decline_rate
staleness_bucket                     
<90d              20655         0.512
90-179d            9171         0.611
180-364d            169         0.467
365d+                 5         0.600
Verdict: MIXED

Signal 2 -- position tier vs mean CTR (impressions_90d >= 100 only):
                  n  mean_ctr
position_tier                
page_1         8633     0.355
top_3           533     0.334
striking       5903     0.256
page_3_5       6058     0.142
deep            879     0.055
Verdict: CONFIRMED


**Reading the MIXED verdict on Signal 1 honestly:** decline rate doesn't rise cleanly with staleness (0.512 -> 0.611 -> 0.467 -> 0.600) -- and the two "very stale" buckets are tiny (n=169 and n=5 out of 30,000), so there isn't enough evidence there to draw a real conclusion either way. This is exactly the "clearly-explained negative" the assignment asks for: staleness alone does **not** cleanly predict decline in this slice. I kept it in the rule anyway, for a different reason than "it predicts decline" -- staleness is the thing an editor can actually act on (you can refresh a stale page; you can't directly un-decline a page), and Signal 2's CONFIRMED result (position -> CTR) is what actually carries the rule's predictive weight. Worth remembering once modeling starts: don't expect `days_since_last_update` alone to be a strong feature here.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os

stale = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible_flag * df["impressions_90d"]  # readable on purpose

flagged = df["score"] > 0
df["reason_code"] = ""
df.loc[flagged, "reason_code"] = "stale_but_visible"
df["action"] = "no_action_needed"
df.loc[flagged, "action"] = "review_for_refresh"

queue_cols = ["content_id", "client_id", "score", "reason_code", "action",
              "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
queue = df[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

n_flagged = int(flagged.sum())
print(f"Ranked queue written: ../outputs/baseline_action_score.csv")
print(f"Total pages: {len(queue):,} | flagged (stale_but_visible): {n_flagged:,} ({n_flagged/len(queue)*100:.1f}%)")
print(f"Score range among flagged: {queue.loc[flagged, 'score'].min():.0f} - {queue.loc[flagged, 'score'].max():.0f}")

Ranked queue written: ../outputs/baseline_action_score.csv
Total pages: 30,000 | flagged (stale_but_visible): 17 (0.1%)
Score range among flagged: 0 - 0


## 3. Top-10 review

The assignment card asks for the top 10 (top-20 is an optional stretch per the skeleton's own header — not required here, so I'm doing the required 10). For each: the action, why it's there, and what would specifically make it wrong.

In [3]:
top10 = queue.head(10).copy()

def what_would_make_it_wrong(row):
    reasons = []
    if row["impressions_90d"] >= 5000:
        reasons.append("if this volume is one seasonal spike rather than steady demand")
    if 0 < row["avg_position"] <= 3:
        reasons.append("if it's already ranking near the top -- 'stale' doesn't mean 'broken' when position is this strong")
    reasons.append("if a sibling page on the same site already absorbed any lost demand (consolidation, not decline)")
    return "; ".join(reasons)

for i, row in top10.iterrows():
    print(f"#{i+1}  {row['content_id']}")
    print(f"     action: {row['action']}  (reason_code={row['reason_code']})")
    print(f"     why: impressions_90d={row['impressions_90d']:.0f}, "
          f"days_since_last_update={row['days_since_last_update']:.0f}, "
          f"avg_position={row['avg_position']:.1f}, trend={row['trend_direction']}")
    print(f"     what would make it wrong: {what_would_make_it_wrong(row)}")
    print()

#1  content_cf56e2e2e282
     action: review_for_refresh  (reason_code=stale_but_visible)
     why: impressions_90d=61678, days_since_last_update=194, avg_position=19.7, trend=down
     what would make it wrong: if this volume is one seasonal spike rather than steady demand; if a sibling page on the same site already absorbed any lost demand (consolidation, not decline)

#2  content_7368877ea310
     action: review_for_refresh  (reason_code=stale_but_visible)
     why: impressions_90d=59472, days_since_last_update=194, avg_position=24.8, trend=down
     what would make it wrong: if this volume is one seasonal spike rather than steady demand; if a sibling page on the same site already absorbed any lost demand (consolidation, not decline)

#3  content_1bfaa38ff26c
     action: review_for_refresh  (reason_code=stale_but_visible)
     why: impressions_90d=25715, days_since_last_update=194, avg_position=22.2, trend=down
     what would make it wrong: if this volume is one seasonal spike rat

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
weak = top10[(top10["avg_position"] > 0) & (top10["avg_position"] <= 3)]
print(f"Weak picks in the top 10: {len(weak)}")
if len(weak):
    print("These rank #1-3 in search already -- the rule flagged them purely for being stale and")
    print("high-traffic, but staleness matters a lot less when a page is already winning its position.")
    print("A sharper rule would down-weight or exclude already-dominant positions, not just chase volume.")
    print(weak[["content_id", "impressions_90d", "avg_position", "days_since_last_update"]].to_string(index=False))
else:
    print("None of the top 10 are already in a dominant position -- no obvious weak pick by that test.")

print("\nLeakage check: the score formula uses only", ["days_since_last_update", "impressions_90d"],
      "-- neither trend_direction nor trend_pct (the label source) was used as a scoring input.")
print("trend_direction only appears in the output CSV as read-only context for the human reviewer.")

Weak picks in the top 10: 0
None of the top 10 are already in a dominant position -- no obvious weak pick by that test.

Leakage check: the score formula uses only ['days_since_last_update', 'impressions_90d'] -- neither trend_direction nor trend_pct (the label source) was used as a scoring input.
trend_direction only appears in the output CSV as read-only context for the human reviewer.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.